# NPDES Violation Analysis

This notebook analyzes wastewater discharge violations reported under the **National Pollutant Discharge Elimination System (NPDES)** using a pre-downloaded violations dataset from EPA ECHO, enriched with facility metadata fetched from the ECHO API.

## Data Architecture

| Source | File | Used In | Purpose |
|--------|------|---------|---------|
| EPA ECHO download tool | one or more `*_NPDES_EFF_VIOLATIONS.csv` files | Data Loading | Primary violations dataset — loaded directly with pandas, one file per state |
| [ECHO API](https://echo.epa.gov/api/echo/cwa_rest_services.get_facility_info) | `facility_lookup.csv` (cached) | Facility Enrichment | Facility metadata (name, SIC code, city, state, lat/long) keyed by NPDES_ID |

Add state CSV files to `DATA_FILES` in the **Configuration** cell to expand coverage. The API cache (`facility_lookup.csv`) is keyed by `NPDES_ID` (nationally unique) and grows automatically as new states are added.

## What this notebook does

1. **Data Loading** — Reads all CSVs in `DATA_FILES`, tags each row with its source state, and concatenates into a single DataFrame.
2. **Facility Enrichment** — Looks up facility metadata from the ECHO API for each unique `NPDES_ID`, caches results to `facility_lookup.csv`, and joins to the violations DataFrame.
3. **Analysis** — Summarizes violation trends, identifies repeat offenders, and computes compliance rates over time.
4. **Visualizations** — Produces presentation-quality charts and interactive Plotly figures for exploring the data.

---
*Primary data: EPA ECHO effluent violations download — [ECHO data downloads](https://echo.epa.gov/tools/data-downloads)*

## Environment Setup

In [ ]:
# Install dependencies (safe to re-run)
import sys
!{sys.executable} -m pip install pandas requests matplotlib seaborn plotly ipywidgets --quiet

In [ ]:
import pandas as pd
import requests
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import plotly.express as px
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display

In [ ]:
# --- Matplotlib / Seaborn defaults ---
plt.rcParams.update({
    'figure.figsize': (12, 6),
    'figure.dpi': 120,
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titlesize': 14,
    'axes.titleweight': 'bold',
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 10,
    'legend.framealpha': 0.8,
    'font.family': 'sans-serif',
})

sns.set_theme(
    style='whitegrid',
    palette='muted',
    font_scale=1.1,
    rc={'axes.spines.top': False, 'axes.spines.right': False},
)

# --- Pandas display options ---
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', 120)
pd.set_option('display.float_format', '{:,.4f}'.format)
pd.set_option('display.colheader_justify', 'left')
pd.set_option('display.show_dimensions', True)

---
## Configuration

*Edit this cell to control which states are loaded and where cached files are stored. All other cells read from these variables — no hardcoded filenames anywhere else in the notebook.*

In [ ]:
# ── Data sources ──────────────────────────────────────────────────────────────
# Add one entry per state CSV downloaded from EPA ECHO.
# Drop files into the data/ folder — filenames must follow: {STATE_ABBR}_NPDES_EFF_VIOLATIONS.csv
DATA_FILES = [
    'data/AZ_NPDES_EFF_VIOLATIONS.csv',
    # 'data/CA_NPDES_EFF_VIOLATIONS.csv',
    # 'data/NV_NPDES_EFF_VIOLATIONS.csv',
]

# ── Cache ──────────────────────────────────────────────────────────────────────
# Facility metadata fetched from the ECHO API is written here after the first run.
FACILITY_CACHE_FILE = 'data/facility_lookup.csv'

# ── ECHO API ───────────────────────────────────────────────────────────────────
ECHO_FACILITY_ENDPOINT = 'https://echodata.epa.gov/echo/dfr_rest_services.get_dfr'

---
## 1. Data Loading

*Reads all CSV files listed in `DATA_FILES` (from the Configuration cell above). Each file is loaded with pandas, tagged with a `SOURCE_STATE` column parsed from its filename, then concatenated into a single `violations` DataFrame. No API calls are made in this section.*

In [ ]:
import os
import re

frames = []
for filepath in DATA_FILES:
    # Parse state abbreviation from filename (e.g. "AZ_NPDES_EFF_VIOLATIONS.csv" → "AZ")
    match = re.match(r'^([A-Z]{2})_', os.path.basename(filepath))
    state = match.group(1) if match else 'UNKNOWN'

    df = pd.read_csv(filepath, low_memory=False)
    df['SOURCE_STATE'] = state
    frames.append(df)

violations = pd.concat(frames, ignore_index=True)

print(f'Loaded {len(violations):,} violation records across {violations["SOURCE_STATE"].nunique()} state(s): '
      f'{sorted(violations["SOURCE_STATE"].unique())}')
violations.head()

---
## 2. Facility Enrichment

Each violation record carries an `NPDES_ID` but no human-readable facility name, location, or
industry classification. This section resolves that by querying EPA's ECHO **DFR (Detailed
Facility Report) API** — one call per unique permit — and caching results to
`facility_lookup.csv`.

**Why cache?** There are 200+ unique permits in Arizona alone. Re-running the notebook should
not hammer the API on every execution, so results are persisted to disk and reused on subsequent
runs. When new states are added to `DATA_FILES`, only the *new* `NPDES_ID` values trigger
additional API calls.

After enrichment, each violation row gains: facility name, SIC code + description, city, state,
latitude, and longitude.


In [ ]:
import time

def _extract_facility(data: dict, npdes_id: str) -> dict:
    """Pull the fields we need from a dfr_rest_services.get_dfr JSON response."""
    results = data.get('Results', {})
    row = {
        'NPDES_ID':     npdes_id,
        'FAC_NAME':     None,
        'SIC_CODE':     None,
        'SIC_DESC':     None,
        'FAC_CITY':     None,
        'FAC_STATE':    None,
        'LATITUDE':     None,
        'LONGITUDE':    None,
    }

    # --- Facility name, city, state, lat/long from Permits list ---
    permits = results.get('Permits', [])
    # Prefer ICIS-NPDES entry whose SourceID matches the permit; fall back to FRS
    icis = next((p for p in permits if p.get('EPASystem') == 'ICIS-NPDES'
                                    and p.get('SourceID') == npdes_id), None)
    frs  = next((p for p in permits if p.get('EPASystem') == 'FRS'), None)
    primary = icis or frs or (permits[0] if permits else {})

    row['FAC_NAME']  = primary.get('FacilityName')
    row['FAC_CITY']  = primary.get('FacilityCity')
    row['FAC_STATE'] = primary.get('FacilityState')

    # FRS entry tends to have better geocoded coordinates
    geo_src = frs or primary
    row['LATITUDE']  = geo_src.get('Latitude')
    row['LONGITUDE'] = geo_src.get('Longitude')

    # --- SIC code: find the source that matches this NPDES_ID ---
    sic_sources = results.get('SIC', {}).get('Sources', [])
    for src in sic_sources:
        for code_entry in src.get('SICCodes', []):
            if code_entry.get('SourceID') == npdes_id:
                row['SIC_CODE'] = code_entry.get('SICCode')
                row['SIC_DESC'] = code_entry.get('SICDesc')
                break
        if row['SIC_CODE']:
            break

    return row


# ── 1. Determine which NPDES_IDs still need API lookup ────────────────────────
import os
import pandas as pd

all_ids = violations['NPDES_ID'].dropna().unique().tolist()

if os.path.exists(FACILITY_CACHE_FILE):
    facility_lookup = pd.read_csv(FACILITY_CACHE_FILE, dtype=str)
    cached_ids = set(facility_lookup['NPDES_ID'].unique())
    missing_ids = [i for i in all_ids if i not in cached_ids]
    print(f"Cache loaded: {len(facility_lookup):,} facilities. "
          f"{len(missing_ids):,} new IDs to fetch.")
else:
    facility_lookup = pd.DataFrame()
    missing_ids = all_ids
    print(f"No cache found. Will fetch {len(missing_ids):,} facilities from ECHO API.")

# ── 2. Fetch missing IDs from ECHO DFR API ────────────────────────────────────
import requests

new_rows = []
errors   = []

for idx, npdes_id in enumerate(missing_ids):
    try:
        resp = requests.get(
            ECHO_FACILITY_ENDPOINT,
            params={'p_id': npdes_id, 'output': 'JSON'},
            timeout=60,
        )
        resp.raise_for_status()
        row = _extract_facility(resp.json(), npdes_id)
    except Exception as exc:
        errors.append(npdes_id)
        row = {'NPDES_ID': npdes_id, 'FAC_NAME': None, 'SIC_CODE': None,
               'SIC_DESC': None, 'FAC_CITY': None, 'FAC_STATE': None,
               'LATITUDE': None, 'LONGITUDE': None}
        print(f"  [WARN] {npdes_id}: {exc}")

    new_rows.append(row)

    if (idx + 1) % 10 == 0:
        print(f"  Fetched {idx + 1}/{len(missing_ids)} ...")

    time.sleep(0.5)

# ── 3. Merge new rows into cache and save ─────────────────────────────────────
if new_rows:
    new_df = pd.DataFrame(new_rows)
    facility_lookup = pd.concat([facility_lookup, new_df], ignore_index=True)
    facility_lookup.to_csv(FACILITY_CACHE_FILE, index=False)
    print(f"Cache updated: {len(facility_lookup):,} total facilities saved to {FACILITY_CACHE_FILE}.")

if errors:
    print(f"\n[WARN] {len(errors)} IDs failed API lookup: {errors}")

# ── 4. Join facility metadata onto violations ─────────────────────────────────
violations = violations.merge(
    facility_lookup[['NPDES_ID', 'FAC_NAME', 'SIC_CODE', 'SIC_DESC',
                      'FAC_CITY', 'FAC_STATE', 'LATITUDE', 'LONGITUDE']],
    on='NPDES_ID',
    how='left',
)

# Coerce lat/long to float for later use in map charts
violations['LATITUDE']  = pd.to_numeric(violations['LATITUDE'],  errors='coerce')
violations['LONGITUDE'] = pd.to_numeric(violations['LONGITUDE'], errors='coerce')

# ── 5. Summary ────────────────────────────────────────────────────────────────
matched   = violations['FAC_NAME'].notna().sum()
total     = len(violations)
fac_count = violations['NPDES_ID'].nunique()
print(f"\nEnrichment complete.")
print(f"  Violation records : {total:,}")
print(f"  Unique facilities : {fac_count:,}")
print(f"  Rows with FAC_NAME: {matched:,} ({matched/total*100:.1f}%)")
violations[['NPDES_ID', 'FAC_NAME', 'SIC_CODE', 'SIC_DESC',
            'FAC_CITY', 'FAC_STATE', 'LATITUDE', 'LONGITUDE']].drop_duplicates('NPDES_ID').head(5)


---
## 3. Analysis

With violations enriched by facility metadata, we can answer three questions that matter to
regulators and the public:

1. **Are violations trending up or down?** — Year-over-year and monthly totals reveal whether
   compliance is improving.
2. **Who are the repeat offenders?** — A small number of facilities often account for a
   disproportionate share of violations.
3. **What parameters are most commonly violated?** — Understanding which pollutants drive the
   most violations informs where monitoring and enforcement attention is best spent.
4. **What types of violations dominate?** — Late DMR submissions vs. numeric exceedances
   have very different regulatory implications.


In [ ]:
import pandas as pd

# ── Date parsing ──────────────────────────────────────────────────────────────
violations['PERIOD_END'] = pd.to_datetime(
    violations['MONITORING_PERIOD_END_DATE'], errors='coerce'
)
violations['YEAR']  = violations['PERIOD_END'].dt.year
violations['MONTH'] = violations['PERIOD_END'].dt.month

# Friendly display name: prefer FAC_NAME, fall back to NPDES_ID
violations['DISPLAY_NAME'] = violations['FAC_NAME'].fillna(violations['NPDES_ID'])

# ── Helper: styled table ───────────────────────────────────────────────────────
def _style(df, caption=''):
    return (
        df.style
          .set_caption(caption)
          .set_table_styles([
              {'selector': 'caption',
               'props': [('font-size', '13px'), ('font-weight', 'bold'),
                         ('text-align', 'left'), ('padding-bottom', '6px')]},
              {'selector': 'th',
               'props': [('background-color', '#2c5f8a'), ('color', 'white'),
                         ('font-size', '11px'), ('text-align', 'left'),
                         ('padding', '6px 10px')]},
              {'selector': 'td',
               'props': [('font-size', '11px'), ('padding', '5px 10px')]},
              {'selector': 'tr:nth-child(even)',
               'props': [('background-color', '#f2f7fc')]},
          ])
          .format(thousands=',')
    )

# ─────────────────────────────────────────────────────────────────────────────
# 3.1  Violation trends by year
# ─────────────────────────────────────────────────────────────────────────────
from IPython.display import display

print("=" * 70)
print("3.1  Violation Trends by Year")
print("-" * 70)
print(
    "Annual violation counts show whether compliance is improving over time. "
    "Years with very few records at the boundary of the dataset reflect partial "
    "monitoring periods rather than genuine compliance improvement."
)

yearly = (
    violations
    .dropna(subset=['YEAR'])
    .query('1995 <= YEAR <= 2024')   # trim sparse boundary years
    .groupby('YEAR', as_index=False)
    .size()
    .rename(columns={'size': 'Violations'})
    .assign(YEAR=lambda d: d['YEAR'].astype(int))
    .set_index('YEAR')
)
display(_style(yearly, caption='Table 1 — Annual Violation Counts (1995–2024)'))

# ─────────────────────────────────────────────────────────────────────────────
# 3.2  Top 15 facilities by total violation count
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 70)
print("3.2  Top 15 Facilities by Violation Count")
print("-" * 70)
print(
    "A handful of facilities consistently account for a large share of all "
    "violations. The table below ranks by total violation count and includes "
    "city and SIC description to aid interpretation."
)

top_facs = (
    violations
    .groupby(['NPDES_ID', 'DISPLAY_NAME', 'FAC_CITY', 'SIC_DESC'], as_index=False)
    .size()
    .rename(columns={'size': 'Total Violations'})
    .sort_values('Total Violations', ascending=False)
    .head(15)
    .reset_index(drop=True)
)
top_facs.index += 1
top_facs.columns = ['NPDES ID', 'Facility Name', 'City', 'Industry', 'Total Violations']
display(_style(top_facs, caption='Table 2 — Top 15 Facilities by Total Violations'))

# ─────────────────────────────────────────────────────────────────────────────
# 3.3  Top 20 most frequently violated parameters
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 70)
print("3.3  Top 20 Most Frequently Violated Parameters")
print("-" * 70)
print(
    "Parameters with the most violations indicate where effluent limits are "
    "most difficult to meet — often due to treatment technology constraints "
    "or highly variable influent loads."
)

top_params = (
    violations
    .dropna(subset=['PARAMETER_DESC'])
    .groupby('PARAMETER_DESC', as_index=False)
    .size()
    .rename(columns={'size': 'Violations'})
    .sort_values('Violations', ascending=False)
    .head(20)
    .reset_index(drop=True)
)
top_params.index += 1
top_params.columns = ['Parameter', 'Violations']
display(_style(top_params, caption='Table 3 — Top 20 Most Frequently Violated Parameters'))

# ─────────────────────────────────────────────────────────────────────────────
# 3.4  Violation type breakdown
# ─────────────────────────────────────────────────────────────────────────────
print()
print("=" * 70)
print("3.4  Violation Type Breakdown")
print("-" * 70)
print(
    "ECHO reports three primary violation codes: D80 (monitoring-only report "
    "overdue), D90 (limit-bearing report overdue), and E90 (numeric effluent "
    "limit exceeded). The mix reveals whether the compliance problem is "
    "primarily reporting failure or actual pollutant exceedances."
)

vtype = (
    violations
    .groupby(['VIOLATION_CODE', 'VIOLATION_DESC'], as_index=False)
    .size()
    .rename(columns={'size': 'Count'})
    .sort_values('Count', ascending=False)
    .reset_index(drop=True)
)
vtype['% of Total'] = (vtype['Count'] / vtype['Count'].sum() * 100).round(1)
vtype.index += 1
vtype.columns = ['Code', 'Description', 'Count', '% of Total']
display(_style(vtype, caption='Table 4 — Violation Type Breakdown'))


---
## 4. Visualizations

The charts below are built with **Plotly Express** so they remain fully interactive on a
static GitHub Pages export — no live Python kernel required. Each chart is designed to stand
alone as a portfolio piece, with descriptive titles, labeled axes, and clean layouts.

- **Chart 1** — Violation counts by year (trend line chart)
- **Chart 2** — Top 15 facilities by violation count (horizontal bar)
- **Chart 3** — Top 20 violated parameters, color-coded by measurement technique
- **Chart 4** — Arizona facility map, dots sized by violation count and colored by SIC category


In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# Shared layout helper
# ─────────────────────────────────────────────────────────────────────────────
def _clean_layout(fig, title, xlab, ylab, height=520):
    fig.update_layout(
        title=dict(text=title, font=dict(size=18, color='#1a1a2e'), x=0, xanchor='left'),
        xaxis_title=xlab,
        yaxis_title=ylab,
        height=height,
        font=dict(family='Arial, sans-serif', size=12, color='#333333'),
        paper_bgcolor='white',
        plot_bgcolor='#f8fbff',
        xaxis=dict(showgrid=True, gridcolor='#dde6f0', zeroline=False),
        yaxis=dict(showgrid=True, gridcolor='#dde6f0', zeroline=False),
        margin=dict(l=60, r=40, t=70, b=60),
    )
    return fig

# ─────────────────────────────────────────────────────────────────────────────
# Chart 1 — Violation counts over time (annual)
# ─────────────────────────────────────────────────────────────────────────────
yearly_plot = (
    violations
    .dropna(subset=['YEAR'])
    .query('1995 <= YEAR <= 2024')
    .groupby('YEAR', as_index=False)
    .size()
    .rename(columns={'size': 'Violations'})
    .assign(YEAR=lambda d: d['YEAR'].astype(int))
)

fig1 = px.line(
    yearly_plot,
    x='YEAR', y='Violations',
    markers=True,
    color_discrete_sequence=['#2c5f8a'],
)
fig1.update_traces(
    line=dict(width=2.5),
    marker=dict(size=7, symbol='circle', line=dict(width=1.5, color='white')),
    hovertemplate='<b>%{x}</b><br>Violations: %{y:,}<extra></extra>',
)
_clean_layout(
    fig1,
    title='NPDES Effluent Violations in Arizona — Annual Trend (1995–2024)',
    xlab='Year',
    ylab='Number of Violations',
)
fig1.show()

# ─────────────────────────────────────────────────────────────────────────────
# Chart 2 — Top 15 facilities by violation count (horizontal bar)
# ─────────────────────────────────────────────────────────────────────────────
top15_plot = (
    violations
    .groupby(['NPDES_ID', 'DISPLAY_NAME'], as_index=False)
    .size()
    .rename(columns={'size': 'Violations'})
    .sort_values('Violations', ascending=False)
    .head(15)
    .sort_values('Violations', ascending=True)   # ascending so largest is at top
)

fig2 = px.bar(
    top15_plot,
    x='Violations', y='DISPLAY_NAME',
    orientation='h',
    color='Violations',
    color_continuous_scale='Blues',
    text='Violations',
)
fig2.update_traces(
    texttemplate='%{text:,}',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Violations: %{x:,}<extra></extra>',
)
fig2.update_coloraxes(showscale=False)
_clean_layout(
    fig2,
    title='Top 15 NPDES Facilities by Total Violation Count — Arizona',
    xlab='Total Violations',
    ylab='',
    height=560,
)
fig2.update_layout(yaxis=dict(showgrid=False))
fig2.show()

# ─────────────────────────────────────────────────────────────────────────────
# Chart 3 — Top 20 parameters, color-coded by measurement technique
# ─────────────────────────────────────────────────────────────────────────────

# ── technique classification ──────────────────────────────────────────────────
ISE_KEYWORDS         = ['ph', 'fluoride', 'nitrate', 'ammonia', 'chloride', 'conductivity']
COLORIMETRIC_KEYWORDS= ['phosphorus', 'chromium, hexavalent', 'ammonia colorimetric']
TITRATION_KEYWORDS   = ['hardness', 'alkalinity', 'cyanide']
IC_KEYWORDS          = ['sulfate', 'nitrite', 'nitrate', 'fluoride', 'chloride']
XRF_KEYWORDS         = ['arsenic', 'copper', 'zinc', 'lead', 'cadmium', 'selenium',
                         'silver', 'nickel', 'chromium']

def _classify_technique(param: str) -> str:
    p = param.lower()
    if any(k in p for k in COLORIMETRIC_KEYWORDS):
        return 'Colorimetric'
    if any(k in p for k in TITRATION_KEYWORDS):
        return 'Titration'
    if any(k in p for k in IC_KEYWORDS):
        return 'Ion Chromatography (IC)'
    if any(k in p for k in ISE_KEYWORDS):
        return 'Ion-Selective Electrode (ISE)'
    if any(k in p for k in XRF_KEYWORDS):
        return 'XRF / Metals'
    return 'Other'

top20_params = (
    violations
    .dropna(subset=['PARAMETER_DESC'])
    .groupby('PARAMETER_DESC', as_index=False)
    .size()
    .rename(columns={'size': 'Violations'})
    .sort_values('Violations', ascending=False)
    .head(20)
    .sort_values('Violations', ascending=True)
)
top20_params['Technique'] = top20_params['PARAMETER_DESC'].apply(_classify_technique)

TECHNIQUE_COLORS = {
    'Ion-Selective Electrode (ISE)': '#2c5f8a',
    'Colorimetric':                  '#27a99e',
    'Titration':                     '#e8a838',
    'Ion Chromatography (IC)':       '#8e44ad',
    'XRF / Metals':                  '#c0392b',
    'Other':                         '#7f8c8d',
}

fig3 = px.bar(
    top20_params,
    x='Violations', y='PARAMETER_DESC',
    orientation='h',
    color='Technique',
    color_discrete_map=TECHNIQUE_COLORS,
    text='Violations',
    category_orders={'Technique': list(TECHNIQUE_COLORS.keys())},
)
fig3.update_traces(
    texttemplate='%{text:,}',
    textposition='outside',
    hovertemplate='<b>%{y}</b><br>Violations: %{x:,}<br>Technique: %{fullData.name}<extra></extra>',
)
_clean_layout(
    fig3,
    title='Top 20 Most Violated Parameters — Colored by Measurement Technique',
    xlab='Total Violations',
    ylab='',
    height=640,
)
fig3.update_layout(
    yaxis=dict(showgrid=False),
    legend=dict(title='Measurement Technique', x=0.55, y=0.08,
                bgcolor='rgba(255,255,255,0.85)', bordercolor='#ccc', borderwidth=1),
)
fig3.show()

# ─────────────────────────────────────────────────────────────────────────────
# Chart 4 — Arizona facility map: dot size = violation count, color = SIC category
# ─────────────────────────────────────────────────────────────────────────────
SIC_CATEGORIES = {
    '49': 'Electric, Gas & Sanitary Services',
    '26': 'Paper & Allied Products',
    '28': 'Chemicals & Allied Products',
    '33': 'Primary Metal Industries',
    '20': 'Food & Kindred Products',
    '10': 'Metal Mining',
    '15': 'Building Construction',
}

def _sic_category(sic):
    if pd.isna(sic):
        return 'Unknown'
    prefix2 = str(sic)[:2]
    return SIC_CATEGORIES.get(prefix2, f'SIC {str(sic)[:2]}xx — Other')

map_df = (
    violations
    .dropna(subset=['LATITUDE', 'LONGITUDE', 'FAC_NAME'])
    .groupby(['NPDES_ID', 'DISPLAY_NAME', 'LATITUDE', 'LONGITUDE', 'SIC_CODE', 'SIC_DESC',
              'FAC_CITY'], as_index=False)
    .size()
    .rename(columns={'size': 'Violations'})
)
map_df['SIC_Category'] = map_df['SIC_CODE'].apply(_sic_category)

fig4 = px.scatter_map(
    map_df,
    lat='LATITUDE',
    lon='LONGITUDE',
    size='Violations',
    color='SIC_Category',
    hover_name='DISPLAY_NAME',
    hover_data={
        'FAC_CITY': True,
        'SIC_DESC': True,
        'Violations': True,
        'LATITUDE': False,
        'LONGITUDE': False,
        'SIC_Category': False,
    },
    size_max=40,
    zoom=6,
    center={'lat': 34.0, 'lon': -111.5},
    labels={'SIC_Category': 'Industry (SIC)'},
    color_discrete_sequence=px.colors.qualitative.Safe,
)
fig4.update_layout(
    title=dict(
        text='Arizona NPDES Facilities — Violation Count by Location and Industry',
        font=dict(size=18, color='#1a1a2e'), x=0, xanchor='left'
    ),
    height=660,
    font=dict(family='Arial, sans-serif', size=12),
    paper_bgcolor='white',
    margin=dict(l=0, r=0, t=60, b=0),
    legend=dict(title='Industry (SIC)', bgcolor='rgba(255,255,255,0.9)',
                bordercolor='#ccc', borderwidth=1),
    map=dict(style='open-street-map'),
)
fig4.show()


---
## Library Versions

In [ ]:
print('Environment ready.')
print(f'  pandas      {pd.__version__}')
print(f'  requests    {requests.__version__}')
print(f'  matplotlib  {matplotlib.__version__}')
print(f'  seaborn     {sns.__version__}')
print(f'  plotly      {plotly.__version__}')
print(f'  ipywidgets  {widgets.__version__}')